In [2]:
import numpy as np
import math  # Added to replace np.math

# --- 1. SIMULATED MARKET DATA SETUP ---
np.random.seed(42)
num_assets = 4
num_days = 1000

# Generate random daily returns for 4 assets (e.g., Stocks, Bonds, Gold, Crypto)
mean_returns = np.array([0.12, 0.05, 0.07, 0.25]) / 252
volatilities = np.array([0.18, 0.05, 0.12, 0.50]) / np.sqrt(252)

simulated_returns = np.random.normal(mean_returns, volatilities, size=(num_days, num_assets))
risk_free_rate = 0.04 / 252

# --- 2. FITNESS FUNCTION (SHARPE RATIO) ---
def calculate_sharpe_ratio(weights):
    weights = weights / np.sum(weights)
    port_returns = np.dot(simulated_returns, weights)
    exp_return = np.mean(port_returns)
    std_dev = np.std(port_returns)

    if std_dev == 0:
        return -9999

    sharpe = (exp_return - risk_free_rate) / std_dev
    return sharpe * np.sqrt(252)

# --- 3. LÉVY FLIGHT GENERATOR ---
def levy_flight(Lambda=1.5):
    # Mantegna's algorithm using standard python math module
    sigma_u = (math.gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2) /
               (math.gamma((1 + Lambda) / 2) * Lambda * 2**((Lambda - 1) / 2)))**(1 / Lambda)

    u = np.random.normal(0, sigma_u, size=num_assets)
    v = np.random.normal(0, 1, size=num_assets)

    step = u / (np.abs(v)**(1 / Lambda))
    return step

# --- 4. CUCKOO SEARCH ALGORITHM ---
def cuckoo_search_portfolio(n=25, Pa=0.25, max_iter=500):
    nests = np.random.dirichlet(np.ones(num_assets), size=n)
    fitness = np.array([calculate_sharpe_ratio(nest) for nest in nests])

    best_idx = np.argmax(fitness)
    G_best_weights = nests[best_idx].copy()
    G_best_fitness = fitness[best_idx]

    for t in range(max_iter):
        for i in range(n):
            step_size = 0.01 * levy_flight()
            new_nest = nests[i] + step_size * (nests[i] - G_best_weights)

            new_nest = np.abs(new_nest)
            new_nest = new_nest / np.sum(new_nest)

            new_fitness = calculate_sharpe_ratio(new_nest)

            j = np.random.randint(0, n)
            if new_fitness > fitness[j]:
                nests[j] = new_nest
                fitness[j] = new_fitness

        sort_idx = np.argsort(fitness)
        num_abandon = int(Pa * n)
        worst_indices = sort_idx[:num_abandon]

        for idx in worst_indices:
            nests[idx] = np.random.dirichlet(np.ones(num_assets))
            fitness[idx] = calculate_sharpe_ratio(nests[idx])

        current_best_idx = np.argmax(fitness)
        if fitness[current_best_idx] > G_best_fitness:
            G_best_fitness = fitness[current_best_idx]
            G_best_weights = nests[current_best_idx].copy()

    return G_best_weights, G_best_fitness

# --- 5. EXECUTION & RESULTS ---
optimal_weights, best_sharpe = cuckoo_search_portfolio()

print("=== OPTIMAL PORTFOLIO ALLOCATION ===")
asset_names = ["Stocks", "Bonds", "Gold", "Crypto"]
for name, weight in zip(asset_names, optimal_weights):
    print(f"{name}: {weight * 100:.2f}%")

print(f"\nMaximized Annualized Sharpe Ratio: {best_sharpe:.4f}")


=== OPTIMAL PORTFOLIO ALLOCATION ===
Stocks: 27.71%
Bonds: 57.31%
Gold: 6.21%
Crypto: 8.77%

Maximized Annualized Sharpe Ratio: 1.4561
